# Turnover vs. refinance

Split SMM into **turnover** (people moving) and **refinance** (people getting a lower rate). The data does not say why a
loan paid off, so the split comes from the model:

1. **Turnover model.** Loans whose rate is well below the market rate (incentive ≤ −0.5 pp) have no reason to
   refinance, so their payoffs are turnover. Train a model on them only: loan age, month, how far below market the loan
   is (lock-in), loan features, and two housing series from FRED.
2. **Apply it to every loan.** Whatever payoff speed is left over is refinance:
   `SMM_refi = SMM − SMM_turn`.
3. **Turnover + Refi model.** Keep turnover fixed and train a refinance model on all loans so that
   `SMM = SMM_turn + SMM_refi` matches the payoffs. Compare it with the Total-SMM Model (the notebook 05 best,
   one model for total SMM).

Incentive is loan rate − PMMS lagged 1 month (the notebook 05 winner).

**Time split (this notebook only):** train 2000–2022, validate 2023, test 2024 on. Notebooks 03–05 train through 2018,
but loans far below the market rate (lock-in) only appear from 2022, so a model trained through 2018 cannot learn how
little those borrowers move. Both models here are trained on this split, so they are compared fairly; their numbers are
not comparable with notebooks 04–05.

In [3]:
# reload source/ modules when they change, no kernel restart needed
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import polars as pl
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss, roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLE = ROOT / "data" / "combined" / "model_table.parquet"  # built by notebook 03
FRED = ROOT / "data" / "fred"                                # FRED series (see README for download links)
sys.path.insert(0, str(ROOT / "source"))
from diagnostics import cpr, cpr_by, check, plot_over_time, shap_summary, permutation_importance

NO_REFI = -0.5  # incentive (pp) at or below which a loan has no reason to refinance
LOAN = ["age", "orig_upb", "fico", "ltv", "cltv", "dti", "num_borrowers", "num_units", "month",
        "first_time_buyer", "loan_purpose", "property_type", "occupancy", "channel", "state", "agency"]
HOUSING = ["hpa_12m", "new_home_sales"]
PARAMS = dict(objective="binary", learning_rate=0.1, num_leaves=63, min_data_in_leaf=2000,  # notebook 04 settings
              feature_fraction=0.8, bagging_fraction=0.5, bagging_freq=1, verbose=-1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Data

- **PMMS** (`MORTGAGE30US`): market rate for the incentive.
- **Case-Shiller home prices** (`CSUSHPINSA`): `hpa_12m` = % change over the past 12 months. Rising prices give owners
  equity to sell and move.
- **New home sales** (`HSN1F`, thousands per year): housing activity. Existing home sales would be better, but FRED only
  has the last 13 months of it; the full history is sold by NAR.

All three are lagged 1 month, so the model only uses what was known before the payoff month.

In [4]:
def fred(series, lag=1):
    """Monthly average of data/fred/<series>.csv (PMMS is weekly), keyed by t (the table's month index) shifted `lag` months forward."""
    date = pl.col("observation_date").str.to_date()
    return (pl.read_csv(FRED / f"{series}.csv", infer_schema=False)
              .select(t=(date.dt.year() * 12 + date.dt.month() - 1 + lag).cast(pl.Int32),
                      value=pl.col(series).cast(pl.Float64, strict=False))  # FRED marks gaps with "."
              .group_by("t").agg(pl.col("value").mean()))

hpi = fred("CSUSHPINSA")
hpa = hpi.join(hpi.select(t=pl.col("t") + 12, prev="value"), on="t").select("t", hpa_12m=100 * (pl.col("value") / pl.col("prev") - 1))

df = (pl.read_parquet(TABLE)
        .join(fred("MORTGAGE30US").rename({"value": "pmms"}), on="t", how="left")
        .join(hpa, on="t", how="left")
        .join(fred("HSN1F").rename({"value": "new_home_sales"}), on="t", how="left")
        .with_columns(incentive=pl.col("orig_rate") - pl.col("pmms"))  # replaces the notebook 03 incentive
        .with_columns(lockin=pl.col("incentive").clip(None, NO_REFI),  # turnover model input; flat above −0.5 pp
                      split=pl.when(pl.col("year") <= 2022).then(pl.lit("train"))  # replaces the notebook 03 split
                              .when(pl.col("year") == 2023).then(pl.lit("valid")).otherwise(pl.lit("test"))))
HELD_OUT = [(2023, 2024, "valid"), (2024, 2027, "test")]  # shaded in the CPR-over-time chart
data = {s: df.filter(pl.col("split") == s) for s in ["train", "valid", "test"]}

**How many loans have no reason to refinance, by year.** This is the sample the turnover model learns from. It is
large when rates have risen (2006–07, 2018, 2022 on) and almost empty in refinance waves (2012, 2020), which is when the
housing series have to carry the turnover level.

In [5]:
pl.Config.set_tbl_rows(30)
(df.group_by("year")
   .agg(loan_months=pl.len(), no_refi=(pl.col("incentive") <= NO_REFI).sum(),
        cpr_all=cpr(pl.col("prepay").mean()), cpr_no_refi=cpr(pl.col("prepay").filter(pl.col("incentive") <= NO_REFI).mean()),
        hpa_12m=pl.col("hpa_12m").mean(), new_home_sales=pl.col("new_home_sales").mean())
   .with_columns(no_refi_pct=100 * pl.col("no_refi") / pl.col("loan_months"))
   .sort("year"))

year,loan_months,no_refi,cpr_all,cpr_no_refi,hpa_12m,new_home_sales,no_refi_pct
i32,u32,u32,f64,f64,f64,f64,f64
2000,265891,16765,5.621999,1.563393,8.87345,880.439067,6.305215
2001,796648,5768,37.022382,1.856414,8.22632,905.374551,0.724034
2002,797402,14594,36.022821,2.117041,7.788434,970.806973,1.830194
2003,607149,11855,49.583122,4.072109,9.372223,1074.5338,1.952568
2004,522087,33043,24.455394,6.036347,12.184973,1192.035921,6.329022
2005,629059,40594,16.2309,6.702131,14.186737,1279.326265,6.453131
2006,818675,311756,9.296756,7.46828,7.590977,1063.039177,38.080557
2007,1077970,271666,8.71698,7.165355,-1.457844,793.553992,25.201629
2008,1335931,149871,9.601047,4.661435,-8.38597,497.791815,11.218469


## 2. Turnover model

Trained on no-refinance loans in the train years, early-stopped on no-refinance loans in the validation years. Two
versions: loan features only, and loan features + housing series. Both are scored on no-refinance loans (lower log loss
is better). The train years now include 2022, the first year with many locked-in loans.

In [6]:
def fit(features, train, valid, params={}, start=None, feval=None):
    """LightGBM with the notebook 04 settings, early-stopped on `valid`. `start`: constant starting score (custom loss only)."""
    def dataset(frame, **kw):
        init = None if start is None else np.full(len(frame), start)
        return lgb.Dataset(frame.select(features).to_pandas(), frame["prepay"].to_numpy(), init_score=init, **kw)
    train_set = dataset(train)
    return lgb.train({**PARAMS, **params}, train_set, num_boost_round=1000, valid_sets=[dataset(valid, reference=train_set)],
                     feval=feval, callbacks=[lgb.early_stopping(50, verbose=False)])

def predict(model, features, frame, **kw):
    return model.predict(frame.select(features).to_pandas(), num_iteration=model.best_iteration, **kw)

def score(name, y, p, w):
    return dict(model=name, log_loss=log_loss(y, p), auc=roc_auc_score(y, p),
                actual_cpr=cpr((y * w).sum() / w.sum()), predicted_cpr=cpr((p * w).sum() / w.sum()))

no_refi = {s: d.filter(pl.col("incentive") <= NO_REFI) for s, d in data.items()}
turn_features = {"loan only": LOAN + ["lockin"], "loan + housing": LOAN + ["lockin"] + HOUSING}
turn_models = {name: fit(f, no_refi["train"], no_refi["valid"]) for name, f in turn_features.items()}

pl.DataFrame([{"split": s, **score(name, no_refi[s]["prepay"].to_numpy(), predict(m, turn_features[name], no_refi[s]),
                                  no_refi[s]["exposure"].to_numpy())}
              for s in ["valid", "test"] for name, m in turn_models.items()])

split,model,log_loss,auc,actual_cpr,predicted_cpr
str,str,f64,f64,f64,f64
"""valid""","""loan only""",0.027121,0.620057,3.831258,5.135708
"""valid""","""loan + housing""",0.027054,0.61833,3.831258,4.192935
"""test""","""loan only""",0.028663,0.58935,4.278111,5.818052
"""test""","""loan + housing""",0.028532,0.591261,4.278111,4.845577


**Turnover check, test years, no-refinance loans only**: actual vs. predicted CPR by how far below market the loan is
and by loan age.

In [7]:
best_turn = min(turn_models, key=lambda n: log_loss(no_refi["valid"]["prepay"].to_numpy(),
                                                     predict(turn_models[n], turn_features[n], no_refi["valid"])))
print("turnover model:", best_turn)
TURN_FEATURES = turn_features[best_turn]
errors, _ = check(no_refi["test"], {name: predict(m, turn_features[name], no_refi["test"]) for name, m in turn_models.items()})
errors

turnover model: loan + housing


model,incentive_pp_error,age_years_error
str,f64,f64
"""loan only""",1.537383,1.535254
"""loan + housing""",0.566161,0.565588


## 3. Split total SMM into turnover and refinance

Apply the turnover model to every loan. The split only makes sense for groups of loans (one payoff is a 0/1 event), so
it is done per month: refinance is what is left of the monthly SMM after turnover. In months where almost every loan
is below market, actual and turnover should nearly match.

In [8]:
for s in data:
    data[s] = data[s].with_columns(turn=predict(turn_models[best_turn], TURN_FEATURES, data[s]))
everything = pl.concat(data.values())

w = pl.col("exposure")
avg = lambda c: (pl.col(c) * w).sum() / w.sum()
by_month = (everything.group_by("t", "year").agg(smm=avg("prepay"), turn=avg("turn"))
                      .with_columns(refi=pl.col("smm") - pl.col("turn"),
                                    date=pl.date(pl.col("t") // 12, pl.col("t") % 12 + 1, 1))
                      .sort("t"))

date, total, turn = by_month["date"].to_numpy(), cpr(by_month["smm"]).to_numpy(), cpr(by_month["turn"]).to_numpy()
fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(date, 0, turn, color="#2a78d6", alpha=0.35, lw=0, label="turnover (model)")
ax.fill_between(date, turn, total, where=total >= turn, interpolate=True, color="#d6602a", alpha=0.35, lw=0,
                label="refinance (actual − turnover)")
ax.plot(date, total, color="#0b0b0b", lw=1.4, label="actual total CPR")
ax.set(title="Monthly CPR split into turnover and refinance", ylabel="CPR (%)", ylim=(0, None))
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

In [9]:
(by_month.group_by("year")
         .agg(cpr=cpr(pl.col("smm").mean()), turnover_cpr=cpr(pl.col("turn").mean()), refi_cpr=cpr(pl.col("refi").mean()),
              months_refi_below_0=(pl.col("refi") < 0).sum())
         .with_columns(refi_share_pct=100 * (1 - pl.col("turnover_cpr") / pl.col("cpr")))
         .sort("year"))

year,cpr,turnover_cpr,refi_cpr,months_refi_below_0,refi_share_pct
i32,f64,f64,f64,u32,f64
2000,4.894542,2.427151,2.523656,1,50.411075
2001,39.7055,5.18104,36.281022,0,86.95133
2002,38.662824,7.417509,33.57075,0,80.81488
2003,52.844601,9.0971,47.847798,0,82.785185
2004,25.402548,8.518475,18.332144,0,66.46606
2005,16.211455,8.434808,8.433357,0,47.970072
2006,8.840974,7.233911,1.721661,0,18.17744
2007,8.639377,6.467662,2.309123,0,25.137403
2008,10.21642,4.61127,5.853674,0,54.864127


## 4. Turnover + Refi model

Turnover is fixed from section 2. A second LightGBM learns refinance (features: loan features + incentive) through the
combined probability `p = p_turn + p_refi`, so it only has to explain the payoffs turnover does not.
LightGBM has no built-in loss for this, so it gets a custom one (log loss of `p`, with gradient and Fisher-scoring
hessian with respect to the refinance score). Compared with the **Total-SMM Model** (the notebook 05 best, one
model for total SMM), retrained here on the same features and split.

In [10]:
REFI_FEATURES = LOAN + ["incentive"]
START = np.log(0.005 / 0.995)  # starting refinance score (logit of 0.5% a month); LightGBM does not save it, so add it back at predict

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def combined(score, turn):
    return np.clip(turn + sigmoid(score), 1e-15, 1 - 1e-15)  # a sum can pass 1; keep the log loss finite

def two_part_loss(turn):
    """Log loss of p = turn + sigmoid(score) for the refinance score."""
    def objective(score, dataset):
        s, p, y = sigmoid(score), combined(score, turn), dataset.get_label()
        ds = s * (1 - s)  # dp/dscore
        return ds * (p - y) / (p * (1 - p)), ds * ds / (p * (1 - p))
    return objective

def combined_log_loss(turn):
    def metric(score, dataset):
        return "log_loss", log_loss(dataset.get_label(), combined(score, turn), labels=[0, 1]), False
    return metric

refi_model = fit(REFI_FEATURES, data["train"], data["valid"], start=START,
                 params=dict(objective=two_part_loss(data["train"]["turn"].to_numpy()), metric="None"),
                 feval=combined_log_loss(data["valid"]["turn"].to_numpy()))
total_model = fit(REFI_FEATURES, data["train"], data["valid"])  # notebook 05 best: one model for total SMM

def preds(frame):
    refi = sigmoid(predict(refi_model, REFI_FEATURES, frame, raw_score=True) + START)
    turn = frame["turn"].to_numpy()
    return {"Total-SMM Model": predict(total_model, REFI_FEATURES, frame), "Turnover + Refi": turn + refi,
            "Turnover Only": turn}

P = {s: preds(d) for s, d in data.items()}
pl.DataFrame([{"split": s, "rounds": (total_model if n == "Total-SMM Model" else refi_model).best_iteration,
               **score(n, data[s]["prepay"].to_numpy(), P[s][n], data[s]["exposure"].to_numpy())}
              for s in ["valid", "test"] for n in ["Total-SMM Model", "Turnover + Refi"]])

/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_1634/1611136342.py:5: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_1634/1611136342.py:5: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_1634/1611136342.py:5: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_1634/1611136342.py:5: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_1634/1611136342.py:5: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_1634/1611136342.py:5: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_1634/161113

split,rounds,model,log_loss,auc,actual_cpr,predicted_cpr
str,i64,str,f64,f64,f64,f64
"""valid""",195,"""Total-SMM Model""",0.027661,0.627731,4.038812,5.138019
"""valid""",138,"""Turnover + Refi""",0.027659,0.622809,4.038812,4.394318
"""test""",195,"""Total-SMM Model""",0.041222,0.710288,9.207873,12.511937
"""test""",138,"""Turnover + Refi""",0.04119,0.710333,9.207873,11.855004


**Test years, all loans.** **Turnover Only** is the turnover part of the Turnover + Refi model; the gap between it and
the Turnover + Refi line is refinance. The error column for Turnover Only is not a fair score (it leaves out refinance),
it is there for reference.

In [11]:
errors, tables = check(data["test"], P["test"])
errors

model,incentive_pp_error,age_years_error
str,f64,f64
"""Total-SMM Model""",2.978688,3.270591
"""Turnover + Refi""",2.328041,2.602776
"""Turnover Only""",4.921502,4.93929


In [12]:
plot_over_time(everything, {n: np.concatenate([P[s][n] for s in data]) for n in ["Total-SMM Model", "Turnover + Refi", "Turnover Only"]},
               held_out=HELD_OUT)

**Predicted vs. actual CPR by year**, both models. Balance-weighted over each year's loan-months, so the test row in
the scoring table above (2024 on) is a mix of the 2024–2026 rows here.

In [13]:
MODELS = ["Total-SMM Model", "Turnover + Refi"]
(cpr_by(everything, "year", {n: np.concatenate([P[s][n] for s in data]) for n in MODELS}, min_rows=1)
   .with_columns(split=pl.when(pl.col("year") <= 2022).then(pl.lit("train"))
                         .when(pl.col("year") == 2023).then(pl.lit("valid")).otherwise(pl.lit("test")))
   .select("year", "split", "actual", *MODELS))

year,split,actual,Total-SMM Model,Turnover + Refi
i32,str,f64,f64,f64
2000,"""train""",6.242991,8.360042,8.720779
2001,"""train""",40.057936,31.808958,32.474031
2002,"""train""",38.67216,26.86394,27.234785
2003,"""train""",53.482371,31.498498,32.681673
2004,"""train""",25.216944,16.371532,18.113417
2005,"""train""",16.121581,11.919236,14.07824
2006,"""train""",8.848365,7.667998,8.813419
2007,"""train""",8.573667,8.976951,9.302989
2008,"""train""",9.993928,11.374262,9.980633


**Where each model misses.** Balance-weighted average miss in CPR points, per factor, test years. Lower is better.

In [14]:
factors = {"fico_bucket": (pl.col("fico") // 40 * 40), "ltv_bucket": (pl.col("ltv") // 10 * 10),
           "orig_year": pl.col("year") - pl.col("age") // 12, "loan_purpose": pl.col("loan_purpose").cast(pl.String),
           "state": pl.col("state").cast(pl.String), "size_bucket": (pl.col("orig_upb") // 100_000 * 100_000)}
test = data["test"].with_columns(**factors)
rows = []
for f in factors:
    d = cpr_by(test, f, {n: P["test"][n] for n in ["Total-SMM Model", "Turnover + Refi"]})
    rows.append({"factor": f, **{n: ((d[n] - d["actual"]).abs() * d["balance"]).sum() / d["balance"].sum()
                                 for n in ["Total-SMM Model", "Turnover + Refi"]}})
pl.DataFrame(rows)

factor,Total-SMM Model,Turnover + Refi
str,f64,f64
"""fico_bucket""",3.30741,2.650659
"""ltv_bucket""",3.321272,2.654848
"""orig_year""",3.256118,2.581356
"""loan_purpose""",3.294364,2.638057
"""state""",3.28562,2.632945
"""size_bucket""",3.324517,2.673662


## 5. What drives the predictions

Two tools, both on the test years:

- **SHAP**: for each loan-month, how much each feature pushes the prediction up or down (in log-odds). `shap_pct` is a
  feature's share of the total push. Shown for each part of Turnover + Refi separately and for the Total-SMM Model.
- **Permutation importance**: shuffle one feature's column, which breaks its link to the payoffs, and measure how much
  worse log loss gets. A bigger increase means the model relies on that feature more for accuracy; about 0 means it
  does not use it. SHAP shows what moves the predictions; permutation shows what makes them right.

In [15]:
shap_tables = {name: shap_summary(model, frame.select(features).to_pandas(), title=f"SHAP, {name} (test years)")
               for name, model, frame, features in [
                   ("turnover model", turn_models[best_turn], no_refi["test"], TURN_FEATURES),  # loans with no reason to refinance
                   ("refinance part", refi_model, data["test"], REFI_FEATURES),
                   ("Total-SMM Model", total_model, data["test"], REFI_FEATURES)]}

**SHAP share (%) per feature.** Empty = the model does not have that feature.

In [16]:
(pl.concat([t.select("feature", "shap_pct", model=pl.lit(name)) for name, t in shap_tables.items()])
   .pivot("model", index="feature", values="shap_pct")
   .sort("Total-SMM Model", descending=True, nulls_last=True))

feature,turnover model,refinance part,Total-SMM Model
str,f64,f64,f64
"""incentive""",null,70.134318,43.087046
"""age""",26.11215,5.771297,17.532701
"""orig_upb""",8.809175,6.978542,7.728946
"""state""",8.132448,4.455503,6.528665
"""month""",7.104144,0.905831,4.563938
"""first_time_buyer""",3.253987,1.467645,4.119481
"""fico""",4.415271,1.986888,2.565775
"""loan_purpose""",0.862248,0.997034,2.119481
"""cltv""",2.24541,1.390851,2.114972


**Permutation importance**, 300,000 test loan-months. For Turnover + Refi, shuffling a feature changes both parts if
both use it (e.g. `age`); `lockin` only feeds turnover and `incentive` only refinance. The turnover model alone is
scored on loans with no reason to refinance. `pct` = share of the total log-loss increase.

In [17]:
ALL = list(dict.fromkeys(TURN_FEATURES + REFI_FEATURES))
sample = data["test"].sample(300_000, seed=0)
no_refi_sample = no_refi["test"].sample(300_000, seed=0)

def turnover(X):
    m = turn_models[best_turn]
    return m.predict(X[TURN_FEATURES], num_iteration=m.best_iteration)

def refinance(X):
    return sigmoid(refi_model.predict(X[REFI_FEATURES], num_iteration=refi_model.best_iteration, raw_score=True) + START)

perm = {
    "Total-SMM Model": permutation_importance(lambda X: total_model.predict(X[REFI_FEATURES], num_iteration=total_model.best_iteration),
                                              sample.select(ALL).to_pandas(), sample["prepay"].to_numpy()),
    "Turnover + Refi": permutation_importance(lambda X: turnover(X) + refinance(X),
                                              sample.select(ALL).to_pandas(), sample["prepay"].to_numpy()),
    "turnover model": permutation_importance(turnover, no_refi_sample.select(TURN_FEATURES).to_pandas(),
                                             no_refi_sample["prepay"].to_numpy()),
}
(pl.concat([t.select("feature", "pct", model=pl.lit(name)) for name, t in perm.items()])
   .pivot("model", index="feature", values="pct")
   .sort("Turnover + Refi", descending=True, nulls_last=True))

feature,Total-SMM Model,Turnover + Refi,turnover model
str,f64,f64,f64
"""incentive""",59.263419,56.151071,null
"""age""",15.087754,14.391962,39.586902
"""state""",4.831599,6.263363,5.683557
"""orig_upb""",5.894812,5.488512,18.355147
"""occupancy""",3.422789,3.623125,0.36677
"""cltv""",3.235258,3.378944,1.987611
"""fico""",3.227722,3.107556,1.820003
"""lockin""",0.0,2.479504,12.072238
"""first_time_buyer""",2.15201,1.86309,4.975256


## 6. Takeaways

Split: train 2000–2022, validate 2023, test 2024 on (this notebook only).

- **Turnover + Refi now beats the Total-SMM Model on the test years.** Log loss 0.041190 vs. 0.041222 (small), but the
  average miss is lower on every chart: by incentive 2.33 vs. 2.98 CPR, by age 2.60 vs. 3.27, and by FICO, LTV,
  vintage, purpose, state and loan size 2.58–2.67 vs. 3.26–3.32. Predicted test CPR 11.9 vs. 12.5 (actual 9.2).
- **Lock-in is learned once 2022 is in the training data.** For loans 3.5–4.5 pp below market, turnover is now
  predicted at 3.4–4.1 CPR (actual 1.9–4.0); before it was flat at about 5.5. 2023 refinance is now about 0 instead
  of −1 CPR.
- **Housing series now matter for turnover.** Test miss 0.57 CPR with them vs. 1.54 without (log loss 0.028532 vs.
  0.028663). Without them the model cannot tell the 2023–26 slowdown in home sales from earlier years.
- **The split looks right.** 2006–07, 2018 and 2023 are mostly turnover; the 2001–03, 2009–13 and 2020–21 waves are
  70–87% refinance. Turnover sits at 4–9 CPR.
- **Still wrong:**
  - Both models predict too much refinancing for loans 1–2 pp above market in the test years (about 37–39 CPR vs.
    26–35 actual). These are mostly young 2023–24 loans; their refinance response is weaker than the train years'.
  - Turnover is about 1 CPR too high for loans 1.5–2.5 pp below market and for loans older than 15 years.
- **What drives each part (SHAP, test years):**
  - Turnover: loan age (26%: new owners rarely move), lock-in (17%: further below market → slower), loan size (9%:
    bigger loans move less), state, home-price growth (8%: faster prices → faster turnover) and month (7%).
  - Refinance: incentive dominates (70%). The refinance part learned on its own that loans below market barely
    refinance. Then state, loan size (bigger loans refinance faster: fixed costs matter less), age (older loans
    slower: burnout), and CLTV (less equity → slower).
- **Permutation importance agrees on the top features** (incentive about 56–59% for both full models, then age).
  Housing series score about 0 here even though SHAP gives them 8% and 5% of turnover: within 2024–26 they barely
  change, so shuffling them across test loans changes little. They set the turnover level across years, which this
  test cannot see.
- **Next:** find why recent in-the-money loans refinance less (loan size, credit, vintage); add home-price equity
  (current LTV) to the refinance model.